# Clean those with different sample rate

We will do resampling

!! Must be run before any new data is selected using E_00_xxx_select.ipynb

In [1]:
from model_dataset import SaShiDatasetManualNorm
from model_dataset import MelSpecTransformDBNoNorm as TheTransform
from paths import *
from model_dataset import Normalizer, DeNormalizer, TokenMap

import pandas as pd
import pickle

In [2]:
transform_configs = {
    "sample_rate": 16000,
    "n_fft": 512,
    "hop_length": 128,
    "n_mels": 96,  
}

model_configs = {
    "input_dim": 96,   # this must equal to n_mels
    "output_dim": 96, 
    "inter_dim_0": 512,
    "dropout": 0.5, 
    "num_layers": 5,
}

train_configs = {
    "batch_size": 128,
    "num_epochs": 100,
    "num_workers": 32,
    "learning_rate": 5e-4,
}

In [3]:
selected_segments = ["L", "R"]
phitype_marker = ["L", "R"]
exp_name = "lari"

t_set = pd.read_csv(os.path.join(src_, f"phi-{exp_name}-{phitype_marker[0]}-guide.csv"))
st_set = pd.read_csv(os.path.join(src_, f"phi-{exp_name}-{phitype_marker[1]}-guide.csv"))

# t_set_sampled = t_set.sample(n=len(st_set))

integrated = pd.concat([t_set, st_set], ignore_index=True, sort=False)
integrated = integrated.sample(frac=1).reset_index(drop=True)

In [4]:
mytrans = TheTransform(sample_rate=transform_configs["sample_rate"], hop_length=transform_configs["hop_length"],
                       n_mels=transform_configs["n_mels"], n_fft=transform_configs["n_fft"])

with open(os.path.join(src_, "no-stress-seg.dict"), "rb") as file:
    # Load the object from the file
    mylist = pickle.load(file)
    mylist = ["BLANK"] + mylist
    mylist = mylist + ["SIL"]

# Now you can use the loaded object
mymap = TokenMap(mylist)

mynorm = Normalizer(Normalizer.norm_mvn_manual)

In [5]:
ds = SaShiDatasetManualNorm(
    src_dir=train_cut_phone_, guide_=integrated, 
    mapper=mymap, transform=mytrans, normalizer=mynorm, 
    noise_fixlength=False, noise_amplitude_scale=0.004, mv_config=None, 
    check_sr=True
)

Resampling ..//src/eng/train-clean-100-cp/78/369/0054/78-369-0054-0087.flac from 22050 to 16000
Resampling ..//src/eng/train-clean-100-cp/6019/3185/0018/6019-3185-0018-0034.flac from 96000 to 16000
Resampling ..//src/eng/train-clean-100-cp/1040/133433/0063/1040-133433-0063-0077.flac from 48000 to 16000
